# Train Guardian's Hausa model (phase 3)

This notebook trains the Hausa speech model for Plateau Eye's AI Emergency Guardian, checks how accurate it is, and publishes it to Hugging Face for the app to download.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (Colab), or *Accelerator → GPU T4* (Kaggle). The free tier is enough.
2. Add two secrets (Colab: the key icon on the left; Kaggle: *Add-ons → Secrets*):
   - `HF_TOKEN`: a Hugging Face token with **write** access to the repo you will publish to.
   - `GITHUB_TOKEN`: only if the Rainbow repository is private. A read-only token is enough.
3. Optional but valuable: Hausa recordings shared from the app (*Guardian → Help build Guardian for Hausa → Share*). Upload each shared set as a .zip in step 3. They are the only recordings with real calls for help in them.

**What it uses:** Google FLEURS Hausa (CC-BY-4.0), your app recordings, and OpenAI Whisper (MIT) as the starting point.

**Time:** about 1–2 hours for `whisper-base` on a T4.

The scripts are the same ones in `training/hausa/` that were run end to end on a CPU before this notebook was written.

## 1. Settings

In [ ]:
# Which Whisper to start from. "openai/whisper-base" is the recommended
# balance; "openai/whisper-tiny" is smaller and faster but less accurate;
# "openai/whisper-small" is more accurate but too big for most phones.
BASE_MODEL = "openai/whisper-base"
EPOCHS = 8
BATCH = 16

# Where to publish, e.g. "your-org/guardian-hausa". The app is pointed at
# https://huggingface.co/<HF_REPO>/resolve/main/manifest.json
HF_REPO = "your-org/guardian-hausa"

# Which branch of the Rainbow repository to take the training code from.
# training/hausa/ reaches "main" only when its pull request is merged; until
# then use that pull request's branch, "kusu/tender-dijkstra-uonely".
BRANCH = "main"


## 2. Code and dependencies

In [ ]:
import os, subprocess
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            return os.environ.get(name)

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/KusuConsult-NG/Rainbow.git" if gh else "https://github.com/KusuConsult-NG/Rainbow.git"
if not os.path.isdir("Rainbow"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "Rainbow"], check=True)
if not os.path.isdir("Rainbow/training/hausa"):
    raise SystemExit(
        f"training/hausa/ is not on the branch '{BRANCH}'. Set BRANCH in step 1 to the "
        "pull request's branch, delete the Rainbow folder (!rm -rf Rainbow), and run this cell again."
    )
os.chdir("Rainbow/training/hausa")
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -r requirements.txt

## 3. Recordings from the app (optional)
Upload one or more .zip files shared from the app. Skip this cell if you have none.

In [ ]:
import glob, zipfile, pathlib
rec_dirs = []
try:
    from google.colab import files
    up = files.upload()
except Exception:
    up = {}  # On Kaggle, add the zips as a dataset and list their folders below instead.
for name in up:
    d = pathlib.Path("recordings") / pathlib.Path(name).stem
    d.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(name).extractall(d)
    rec_dirs.append(str(d))
# Kaggle example: rec_dirs = ["/kaggle/input/hausa-recordings-1"]
print("recording folders:", rec_dirs)

## 4. Prepare the data

In [ ]:
rec_flag = ("--recordings " + " ".join(rec_dirs)) if rec_dirs else ""
!python prepare_data.py --out data {rec_flag}

## 5. Train

In [ ]:
!python train.py --data data --model {BASE_MODEL} --out runs/hausa --epochs {EPOCHS} --batch {BATCH}

## 6. Convert and export to the app's format

In [ ]:
!python convert_to_openai.py --hf runs/hausa --out runs/hausa/hausa.pt
!mkdir -p export && cd export && python ../export_onnx.py --model hausa --checkpoint ../runs/hausa/hausa.pt
!ls -la export

## 7. Measure it

Word error rate on FLEURS test (sentences no training ever saw), then the calls for help held back from your recordings, if any.

**How to read it.** For Guardian the distress rules only need the key words right ("kashe ni", "an sace ni"…), so a model can be useful above a perfect score. But as a rough guide:
- **WER under 0.30:** worth trying on phones.
- **0.30–0.50:** expect missed calls; add recordings and train again.
- **Over 0.50:** do not publish.

Look at the transcripts of the held-back calls for help too: they matter more than the overall number.

In [ ]:
!python evaluate.py --model-dir export --manifest data/test.jsonl --report export/report.json
import os
if os.path.getsize("data/test_distress.jsonl") > 0:
    !python evaluate.py --model-dir export --manifest data/test_distress.jsonl --show

## 8. Publish to Hugging Face
Run this only if step 7 says the model is good enough. `publish.py` refuses to upload when word error is above 0.50; it says so and uploads nothing. The manifest it writes lists what the model was trained on, including how many app recordings, if any.

In [ ]:
os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
!python publish.py --model-dir export --repo {HF_REPO} --report export/report.json --base-model {BASE_MODEL} --data data

## 9. Point the app at it

Send the address printed above (`https://huggingface.co/<repo>/resolve/main/manifest.json`) to whoever builds the app. They put it in the app's `.env` before building:

```
GUARDIAN_HAUSA_MANIFEST_URL=https://huggingface.co/<repo>/resolve/main/manifest.json
```

The Guardian screen then offers *Download Hausa*. The app downloads the files listed in the manifest only when a person asks for them, and refuses any file whose size or SHA-256 does not match. Leave the variable blank and Guardian offers English and Pidgin only.
